In [0]:
from time import sleep
from datetime import datetime
from pathlib import Path
from pyspark.sql.functions import lit
from delta.exceptions import ConcurrentAppendException
import boto3
import pandas as pd
from cdp_dq_framework.models import EntityContract
from cdp_dq_framework.utils import read_csv, read_excel, read_parquet

In [ ]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
service_credential = dbutils.widgets.get("service_credential")

In [0]:
# Retrieve task values
tv = dbutils.jobs.taskValues
bucket = tv.get(taskKey="derive_config", key="bucket", debugValue=None)
file_key = tv.get(taskKey="derive_config", key="file_key", debugValue=None)
app_id = tv.get(taskKey="derive_config", key="app_id", debugValue=None)
entity_id = tv.get(taskKey="derive_config", key="entity_id", debugValue=None)
entity_name = tv.get(taskKey="derive_config", key="entity_name", debugValue=None)
excn_id = tv.get(taskKey="derive_config", key="excn_id", debugValue=None)
service_type = tv.get(taskKey="derive_config", key="service_type", debugValue=None)
file_name = tv.get(taskKey="derive_config", key="file_name", debugValue=None)
file_arrival_time = tv.get(taskKey="derive_config", key="file_arrival_time", debugValue=None)
pre_contract_path = tv.get(taskKey="derive_config", key="pre_contract_path", debugValue=None)
pre_contract_file_path = tv.get(taskKey="derive_config", key="pre_contract_file_path", debugValue=None)
post_process_path = tv.get(taskKey="derive_config", key="post_process_path", debugValue=None)
topic_arn = tv.get(taskKey="derive_config", key="topic_arn", debugValue=None)
file_ext = tv.get(taskKey="contract_checks", key="file_ext", debugValue=None)
result_status = tv.get(taskKey="contract_checks", key="result_status", debugValue=None)
result = tv.get(taskKey="contract_checks", key="result", debugValue=None)

In [0]:
now = datetime.now()
file_arrival_time = datetime.strptime(file_arrival_time, '%Y-%m-%d %H:%M:%S')
max_retries = 5
read_purpose = "lake_load"
base = "/".join(pre_contract_path.split("pre_contract")[:-1]).strip("/")
archive_file_path = f"s3://{base}/archive/source/{now.strftime('%Y-%m-%d')}/{excn_id}_{file_name}"

In [0]:
# Email notification function
def send_msg(topic_arn, msg, sub):
    region = topic_arn.split(":")[3]
    session = boto3.Session(
        botocore_session=dbutils.credentials.getServiceCredentialsProvider(service_credential),
        region_name=region
    )
    sns_client = session.client('sns')

    # Keeping subject less than 100 characters
    print(f"SNS subject: {sub[:100]}\nSNS MSG: {msg[:100]}\nTopic ARN used: {topic_arn}")

    # publish the msg to the passed topic ARN
    response = sns_client.publish(
        TopicArn=topic_arn,
        Message=msg,
        Subject=sub,
    )
    return response

# Email notification constants
success_subject = f"CDP validation succeeded for file {file_name}"
success_msg = f"Hi Team,\n\nBelow is the validation statistics for the file {file_name}.\n\n"
greetings = "\n\nRegards,\nCDP"

In [0]:
# copy file from pre_contract to post_contract
if result_status == "SUCCESS":
    post_process_prefix = f"{app_id}_{entity_id}"
    post_process_filename = f"{app_id}_{entity_id}_{excn_id}_{file_name}"
    post_process_file_path = f"s3://{post_process_path}/{post_process_filename}"
    file_name_audit = "_".join(post_process_filename.split("_")[3:])
    files = dbutils.fs.ls(f"s3://{post_process_path}")
    matching_files = [f.path for f in files if f.name.startswith(post_process_prefix) and "*" not in f.path]
    print(f"Source Pre Contract: {pre_contract_file_path}")
    print(f"Target Post Contract: {post_process_file_path}")
    print("Copying file to post-contract path")
    dbutils.fs.cp(pre_contract_file_path, post_process_file_path)  # -> post_contract
    print(f"Archiving pre-contract file to {archive_file_path}")
    dbutils.fs.mv(pre_contract_file_path, archive_file_path)  # archive source
    if matching_files:
        print(f"Deleting {len(matching_files)} file(s) with prefix '{post_process_prefix}'")
        for file_path in matching_files:
            print(f"Deleting {file_path}")
            dbutils.fs.rm(file_path)
    duration = int((now - file_arrival_time).total_seconds())
    spark.sql(f"INSERT INTO {catalog}.{schema}.dc_excn_stat VALUES ('{excn_id}', '{app_id}', '{entity_id}', '{file_name}', 'passed', '{file_arrival_time}', '{now}', '{duration}', '{post_process_path}')")
    print("Audit table updated with status 'passed'")

    # Send email notification
    for chk in result["checks"]:
        success_msg += f"{chk[0]}:\t{chk[1]}\n"
    success_msg += greetings
    send_msg(topic_arn, success_msg, success_subject)

In [0]:
query = f"SELECT * FROM {catalog}.{schema}.dc_load_strategy WHERE UPPER(app_id)='{app_id}' AND UPPER(entity_id)='{entity_id}'"
print(f"{query=}")
lrow = spark.sql(query).limit(1).collect()[0].asDict()

In [0]:
load_type = lrow["load_typ"].strip().casefold()
lake_path = lrow["lake_target_file_path"].strip()
s3_lake_path = f"s3://{lake_path}"

In [0]:
if load_type == "bypassed":  # plain copy, no transform
    print(f"Source Pre Contract: {pre_contract_file_path}")
    print(f"Target Lake Path: {s3_lake_path}")
    print("Moving file to lake path")
    dbutils.fs.cp(pre_contract_file_path, s3_lake_path)
    spark.sql(f"UPDATE {catalog}.{schema}.dc_excn_stat SET status='ByPassed', destination_dir='{lake_path}' WHERE excn_id='{excn_id}'")
    print("Audit table updated with status 'ByPassed'")
    dbutils.notebook.exit("BYPASSED")

In [0]:
query = f"SELECT * FROM {catalog}.{schema}.dc_entity_mstr WHERE UPPER(app_id)='{app_id}' AND UPPER(entity_id)='{entity_id}'"
print(f"{query=}")
row = spark.sql(query).limit(1).collect()[0].asDict()
contract = EntityContract.from_row(row)

In [0]:
# Read file data
if file_ext in ("csv", "txt", "flat", "dat", "tab"):
    file_data_df = read_csv(spark, post_process_file_path, contract, read_purpose, service_type)
elif file_ext in ("xlsx", "xls"):
    file_data_df = read_excel(spark, post_process_file_path, contract, read_purpose, service_type)
else:
    raise Exception(f"Invalid file received: {file_name}")

file_data_df = (file_data_df
    .withColumn("aud_batch_load_wid", lit(now.strftime("%Y%m%d%H%M%S")))
    .withColumn("aud_file_nm", lit(file_name_audit))
    .withColumn("aud_load_dts", lit(now.strftime("%d/%m/%Y %H:%M")))
    .withColumn("aud_row_status_ind", lit('0'))
    .withColumn("aud_src_sys_id", lit(app_id))
    .withColumn("aud_upd_dts", lit(now.strftime("%d/%m/%Y %H:%M")))
)

In [0]:
file_data_df.show(5)

In [0]:
lake_path_file_name = f"{s3_lake_path}/{entity_name}{now.strftime("%Y%m%d%H%M%S%f")}.parquet"
print(f"Writing to {s3_lake_path}")
if load_type == "truncate":
    try:
        objects = dbutils.fs.ls(s3_lake_path)  # List all objects in the s3 lake path
        parquet_files = [obj for obj in objects if obj.name.endswith(".parquet")]  # Filter to only .parquet files
        # Display and delete iteratively
        for obj in parquet_files:
            print(f"Deleting: {obj.name}")
            dbutils.fs.rm(obj.path, recurse=False)
    except Exception as e:
        if "java.io.FileNotFoundException" in str(e) or "CloudFileNotFoundException" in str(e):
            print(f"{s3_lake_path} does not exist yet, will be created on write")
        else:
            raise e
    # Save DF to Target Lake Path
    file_data_df.write.mode("overwrite").parquet(s3_lake_path)
elif load_type == "append":
    # Save DF to Target Lake Path
    file_data_df.coalesce(1).write.mode("append").parquet(s3_lake_path)
elif load_type in ("upsert" , "rolling"):
	# keys = contract["upsert_key_list"] or [df.columns[0]]
	# if not spark.catalog.tableExists(target_table):
	# 	df.write.mode("overwrite").saveAsTable(target_table)
	# else:
	# 	df.createOrReplaceTempView("_incoming")
	# 	on = " AND ".join(f"t.`{k}`=s.`{k}`" for k in keys)
	# 	spark.sql(f"""MERGE INTO {target_table} t USING _incoming s ON {on}
    #         WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
    prev_file_path = next(Path(s3_lake_path).glob(f"{entity_name}*.parquet"))
    prev_data_df = read_parquet(spark, prev_file_path)
    # merged_data_df = pd.concat([file_data_df, prev_data_df], ignore_index=True)
    # merged_data_df = merged_data_df.drop_duplicates(keep='first', subset=contract.upsert_key_list)
    merged_data_df = file_data_df.union(prev_data_df)
    # merged_data_df.to_parquet(lake_path_file_name, engine="auto")
    merged_data_df.coalesce(1).write.mode("append").parquet(s3_lake_path)
    del prev_data_df
    del file_data_df
else:
    raise Exception(f"Invalid load type: {load_type}")

for attempt in range(1, max_retries + 1):
    try:
        spark.sql(f"UPDATE {catalog}.{schema}.dc_excn_stat SET status='moved to lake', destination_dir='{lake_path}' WHERE excn_id='{excn_id}'")
        print("Audit table updated with status 'moved to lake'")
        break
    except ConcurrentAppendException as e:
        if attempt == max_retries:
            raise Exception(f"Max retries reached. Last exception: {e}")
        sleep(2 ** attempt)